# Week 4: A neural network from scratch

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GDG-TUM/AI-ML-Track-GDG-on-Campus-TUM/blob/main/weekly-sessions/week-04-neural-networks-from-scratch/neural_net_from_scratch.ipynb)

**GDG on Campus TUM · AI/ML Track**

Frameworks like Keras and PyTorch are wonderful, but they can feel like magic. Today we remove the magic: a complete neural network, **trained with nothing but NumPy**, in a few dozen lines.

By the end you will have seen every idea that powers modern deep learning:

| Idea | Where you will see it |
|------|-----------------------|
| **Weights and biases** | The numbers the network learns |
| **Activation function** | The bend that lets networks learn curves, not just lines |
| **Loss** | One number that says how wrong we are |
| **Gradient** | Which way to nudge each weight to be less wrong |
| **Backpropagation** | Computing all those nudges efficiently with the chain rule |
| **Gradient descent** | Repeat the nudge thousands of times |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)

## The problem: two interlocking moons

A straight line cannot separate these two classes. A neural network can bend.

In [ ]:
X, y = make_moons(n_samples=500, noise=0.2, random_state=0)
y = y.reshape(-1, 1)  # shape (n, 1) so the maths lines up
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

plt.figure(figsize=(5, 4))
plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap="coolwarm", edgecolor="white")
plt.title("Two classes, one curvy boundary")
plt.show()

## The network

```
input (2 numbers) → hidden layer (16 neurons, tanh) → output (1 neuron, sigmoid) → probability of class 1
```

* Each layer does `output = activation(inputs @ weights + bias)`
* **tanh** squashes numbers into (-1, 1). **sigmoid** squashes into (0, 1), so we can read the result as a probability.
* The **loss** is binary cross-entropy: it is large when the network is confidently wrong.

In [ ]:
def init_params(n_in, n_hidden, n_out, rng):
    return {
        "W1": rng.normal(0, 1 / np.sqrt(n_in), (n_in, n_hidden)),
        "b1": np.zeros((1, n_hidden)),
        "W2": rng.normal(0, 1 / np.sqrt(n_hidden), (n_hidden, n_out)),
        "b2": np.zeros((1, n_out)),
    }


def forward(X, p):
    a1 = np.tanh(X @ p["W1"] + p["b1"])  # hidden layer
    a2 = 1 / (1 + np.exp(-(a1 @ p["W2"] + p["b2"])))  # output probability
    return a2, (X, a1, a2)


def loss_fn(a2, y):
    a2 = np.clip(a2, 1e-12, 1 - 1e-12)  # never take log(0)
    return -np.mean(y * np.log(a2) + (1 - y) * np.log(1 - a2))

## Backpropagation: the chain rule, in code

We want the **gradient** of the loss with respect to every weight: "if I nudge this weight up a tiny bit, does the loss go up or down, and by how much?"

Work backwards from the loss, one layer at a time:

1. Output error: `a2 - y` (a lovely simplification when sigmoid meets cross-entropy)
2. Push it back through the output weights to find each hidden neuron's share of the blame
3. Multiply by the slope of tanh, which is `1 - a1²`
4. Each weight's gradient is "input × blame"

In [ ]:
def backward(y, p, cache):
    X, a1, a2 = cache
    n = X.shape[0]
    dz2 = (a2 - y) / n  # error at the output
    grads = {
        "W2": a1.T @ dz2,
        "b2": dz2.sum(axis=0, keepdims=True),
    }
    dz1 = (dz2 @ p["W2"].T) * (1 - a1**2)  # blame pushed back through tanh
    grads["W1"] = X.T @ dz1
    grads["b1"] = dz1.sum(axis=0, keepdims=True)
    return grads

### Trust, but verify: a gradient check

How do we know our backprop has no bug? Compare it with the slow, obvious way: nudge one weight by a tiny `±h`, measure how the loss changes, and see if it matches. If these agree, the maths is right. This trick has saved careers.

In [ ]:
p = init_params(2, 16, 1, rng)
a2, cache = forward(X_train, p)
grads = backward(y_train, p, cache)

h = 1e-6
worst = 0.0
for name in p:
    for _ in range(5):  # check 5 random entries of each parameter
        i = tuple(rng.integers(0, s) for s in p[name].shape)
        original = p[name][i]
        p[name][i] = original + h
        up = loss_fn(forward(X_train, p)[0], y_train)
        p[name][i] = original - h
        down = loss_fn(forward(X_train, p)[0], y_train)
        p[name][i] = original
        numeric = (up - down) / (2 * h)
        analytic = grads[name][i]
        rel_err = abs(numeric - analytic) / max(1e-8, abs(numeric) + abs(analytic))
        worst = max(worst, rel_err)

print(f"worst relative error over 20 checks: {worst:.2e}")
assert worst < 1e-4, "backprop does not match numerical gradients"
print("✅ Our backpropagation matches the numerical gradient.")

## Training = nudge, repeat

**Gradient descent:** move every weight a small step (the **learning rate**) *against* its gradient. Do it a few thousand times and watch the loss fall.

In [ ]:
def train(X, y, n_hidden=16, lr=1.0, epochs=3000, seed=0, verbose=True):
    rng = np.random.default_rng(seed)
    p = init_params(X.shape[1], n_hidden, 1, rng)
    history = []
    for epoch in range(epochs):
        a2, cache = forward(X, p)
        history.append(loss_fn(a2, y))
        grads = backward(y, p, cache)
        for name in p:
            p[name] -= lr * grads[name]
        if verbose and epoch % 500 == 0:
            print(f"epoch {epoch:5d}  loss {history[-1]:.4f}")
    return p, history


def accuracy(X, y, p):
    return float(((forward(X, p)[0] > 0.5) == y).mean())


params, history = train(X_train, y_train)
print()
print(f"train accuracy: {accuracy(X_train, y_train, params):.3f}")
print(f"test  accuracy: {accuracy(X_test, y_test, params):.3f}")
assert accuracy(X_test, y_test, params) > 0.9

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history, color="#4285F4")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].set_title("The loss falls as the network learns")

xx, yy = np.meshgrid(np.linspace(-2, 3, 300), np.linspace(-1.5, 2, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = forward(grid, params)[0].reshape(xx.shape)
axes[1].contourf(xx, yy, zz, levels=20, cmap="coolwarm", alpha=0.6)
axes[1].scatter(X_test[:, 0], X_test[:, 1], c=y_test.ravel(), cmap="coolwarm", edgecolor="white")
axes[1].set_title("The learned boundary bends around the moons")

plt.tight_layout()
plt.show()

You just trained a neural network. Every framework you will meet (Keras, PyTorch, JAX) does exactly this, only with automatic gradients, GPUs, and many more layers.

## 🏁 Take-home challenge

Experiment. Predict what will happen *before* you run each change.

1. **Hidden size:** try `n_hidden` of 1, 2, 4, 64. What is the smallest network that still solves the problem?
2. **Learning rate:** try `lr` of 0.01, 0.1, 5. What happens to the loss curve when it is too small? Too big?
3. **Activation:** replace `tanh` with ReLU, `np.maximum(0, z)`. You must change the backward pass too: what is the slope of ReLU?
4. **Overfit it:** use only 20 training points and 500 hidden neurons. Compare train and test accuracy (Week 3 in action).
5. Watch [3Blue1Brown's neural network series](https://www.3blue1brown.com/topics/neural-networks), then explain backpropagation to a friend in under two minutes.

**Next week:** the same ideas with a real framework, a GPU, and a pretrained model.